# 🌐 Notebook 3: Heartbeats in the Real World

We've talked about *how to decide* a node is dead. This notebook is about **how heartbeats actually get plumbed into a system**. The same `φ > 8` decision rule looks very different if you build it as push vs pull, central vs gossip.

## Learning objectives
- Compare **push** (sender pushes heartbeats) vs **pull** (monitor probes).
- Compare **central monitor** vs **all-pairs gossip**.
- See why declaring a node dead is only step 1 — you also need **fencing** to avoid a split brain.

## 1. Push vs Pull

**Push** is what we simulated so far: every node sends `"I'm alive"` to a monitor on a timer. Pros:
- Cheap when nodes are healthy (one message per node per interval).
- The monitor's work is bounded: it just listens.

Cons:
- A *silent* node looks the same as one that crashed *and* one whose network dropped its packets *and* one that's just GC-paused.

**Pull** flips it: the monitor sends `"are you there?"` and waits for a reply (think `ping`, HTTP `/healthz`, gRPC health check, Kubernetes liveness probe).

Pros:
- The monitor controls the cadence and can do *active* checks (e.g. "hit the DB and tell me you got rows back").
- Failure includes the cause: timeout vs connection refused vs HTTP 500.

Cons:
- More work for the monitor (one round-trip per node per interval).
- Doesn't scale to thousands of nodes from a single point.

Most real systems use **both**:
- Kubernetes: kubelet **pushes** node status to the API server, while the kubelet itself **pulls** liveness probes from each pod.
- HDFS: DataNode **pushes** heartbeats to the NameNode.
- Load balancers: **pull** `/healthz` from each backend.

Below is a tiny in-memory simulation of both styles so you can feel the difference.

In [ ]:
import random
random.seed(1)

class Network:
    """Toy network. Each call to deliver() flips a coin to drop the message."""
    def __init__(self, drop_rate=0.0):
        self.drop_rate = drop_rate
        self.delivered = 0
        self.dropped = 0
    def deliver(self, msg, handler):
        if random.random() < self.drop_rate:
            self.dropped += 1
            return False
        self.delivered += 1
        handler(msg)
        return True

class Monitor:
    def __init__(self):
        self.last_seen = {}
    def on_heartbeat(self, msg):
        node, t = msg
        self.last_seen[node] = t

def push_simulation(nodes, total=10.0, interval=1.0, drop_rate=0.1):
    net = Network(drop_rate)
    mon = Monitor()
    t = 0.0
    while t < total:
        for n in nodes:
            net.deliver((n, t), mon.on_heartbeat)
        t += interval
    return net, mon

def pull_simulation(nodes, total=10.0, interval=1.0, drop_rate=0.1):
    net = Network(drop_rate)
    mon = Monitor()
    t = 0.0
    while t < total:
        for n in nodes:
            # The monitor sends a probe (1 msg) and the node replies (1 msg).
            # We need BOTH to succeed to count it as a successful health check.
            probe_ok = net.deliver(('probe', n, t), lambda _m: None)
            if probe_ok:
                net.deliver((n, t), mon.on_heartbeat)
        t += interval
    return net, mon

nodes = [f'n{i}' for i in range(20)]
push_net, push_mon = push_simulation(nodes)
pull_net, pull_mon = pull_simulation(nodes)
print(f'PUSH: {push_net.delivered} delivered, {push_net.dropped} dropped, '
      f'{len(push_mon.last_seen)}/{len(nodes)} nodes ever seen')
print(f'PULL: {pull_net.delivered} delivered, {pull_net.dropped} dropped, '
      f'{len(pull_mon.last_seen)}/{len(nodes)} nodes ever seen')

push_total = push_net.delivered + push_net.dropped
pull_total = pull_net.delivered + pull_net.dropped
print(f'\nmessages on the wire: push={push_total}, pull={pull_total} '
      f'({pull_total / push_total:.1f}x)')

# Pull costs a probe AND a reply per check, so it is ~2x the traffic for the same
# coverage — and a drop on either leg loses the check.
assert push_total == len(nodes) * 10          # 20 nodes x 10 rounds, one message each
assert 1.5 < pull_total / push_total <= 2.0, pull_total / push_total
# Effective loss compounds: a 10% per-message drop becomes ~19% per health check.
push_loss = push_net.dropped / push_total
pull_effective_loss = 1 - (pull_net.delivered / 2) / (len(nodes) * 10)
print(f'per-message drop rate ~{push_loss:.0%}; effective pull check-failure rate '
      f'~{pull_effective_loss:.0%} (two legs, both must survive)')
assert pull_effective_loss > push_loss

Notice that **pull roughly doubles the message volume** for the same coverage (probe + reply), and a single drop on either leg counts as a missed probe. That's why push dominates intra-cluster heartbeating, while pull dominates external health-checking (load balancers, k8s probes) where the checker explicitly *wants* control.

## 2. Central monitor vs all-pairs gossip

**Central monitor** (HDFS NameNode, Kubernetes API server): every node reports to one place. Easy to reason about; the monitor is a single source of truth — and a single point of failure.

**All-pairs gossip** (Cassandra, Consul Serf, Akka cluster): every node periodically tells a few random peers what it knows about everyone. There's no central authority. It scales beautifully but the membership view is *eventually* consistent — different nodes can briefly disagree about who's alive.

The cost difference is dramatic. Below we count messages per round.

In [ ]:
def central_msg_count(N):
    # Every node sends 1 heartbeat to the central monitor.
    return N

def gossip_msg_count(N, fanout=3):
    # Every node sends to `fanout` random peers.
    return N * fanout

header = f"{'N':>5} | {'central msgs/round':>18} | {'gossip msgs/round (fanout=3)':>28}"
print(header)
print('-' * len(header))
for N in (10, 100, 1000, 10000):
    print(f'{N:>5} | {central_msg_count(N):>18} | {gossip_msg_count(N):>28}')

# Both are linear in N; gossip pays a constant factor for having no coordinator.
# The naive all-pairs alternative is quadratic, which is the thing to avoid.
def all_pairs(N):
    return N * (N - 1)

print(f"\n{'N':>6} {'central':>10} {'gossip f=3':>12} {'all-pairs':>12}")
for N in (10, 100, 1000, 10000):
    print(f'{N:>6} {central_msg_count(N):>10} {gossip_msg_count(N):>12} {all_pairs(N):>12}')

for N in (10, 100, 1000, 10000):
    assert gossip_msg_count(N) == 3 * central_msg_count(N)     # constant factor
    assert all_pairs(N) >= gossip_msg_count(N)
assert all_pairs(10000) / gossip_msg_count(10000) > 300
print('\n✔ central and gossip are both O(N) per round; all-pairs is O(N²) and is'
      f' {all_pairs(10000) // gossip_msg_count(10000)}x worse at N=10,000')
print('  The reason to pick gossip is the absent coordinator, not the message count.')

Central scales O(N), gossip scales O(N · fanout) per round. Both are way better than the naive O(N²) of *"every node pings every other node directly"*, which is why nobody does that past a handful of nodes.

But the real reason to choose gossip isn't bandwidth — it's that there is **no single coordinator to fail**. The membership view propagates through the cluster the way rumours spread on a playground.

## 3. Step 1 was easy. Step 2 ("now what?") is where it gets hard

Detecting that a node *seems* dead is the easy part. What you do next depends on the system, and getting it wrong is how clusters lose data.

### Common follow-up actions

1. **Stop sending it traffic** — load balancer takes the backend out of rotation.
2. **Re-replicate its data** — HDFS makes a new copy of every block the dead DataNode held; Cassandra streams its token range to a replacement.
3. **Trigger a leader election** — the cluster picks a new primary (Raft, ZooKeeper).
4. **Fence the suspected node** — explicitly cut it off so a *false positive* can't cause damage.

### Why fencing matters: the split brain

Imagine our `φ > 8` detector wrongly suspects a healthy primary database. We promote a new primary. Now there are **two** primaries, each thinking it's the only one, both accepting writes. Welcome to a split brain — manual data reconciliation is in your future.

Fencing prevents this by guaranteeing the suspected-dead node *cannot* keep writing once a new one takes over. Common techniques:

- **STONITH** ("shoot the other node in the head") — power-cycle it via IPMI before failing over.
- **Lease + epoch numbers** — every primary writes with a token that's only valid until the lease expires; the storage layer rejects writes from old epochs.
- **Quorum** — only allow writes if a majority of nodes acknowledge. A network-partitioned minority physically cannot accept writes.

These are big enough topics to deserve their own labs in this repo (see `02-distributed-primitives/lease/`, `02-distributed-primitives/quorum/`, `02-distributed-primitives/split-brain-and-fencing/`).

## 🧠 Cheat sheet

| Question | Rule of thumb |
|---|---|
| How often to heartbeat? | Often enough that `k * interval` < acceptable detection time, but not so often that you burn bandwidth or batteries. 1–10s is typical inside a datacenter. |
| Push or pull? | Push for intra-cluster (cheap, scales). Pull for external health checks (you want control + diagnosis). |
| Central or gossip? | Central is simpler; use gossip when N is large or you can't tolerate a coordinator failure. |
| What threshold? | Don't tune timeouts; tune φ (notebook 2). Cassandra defaults to `φ = 8`, Akka to `12`. |
| What to do on detection? | Stop traffic → re-replicate → re-elect → **fence**. Never skip fencing. |

## 📚 Where to go next in this repo

- `02-distributed-primitives/phi-accrual-failure-detection/` — deep dive on the detector itself.
- `02-distributed-primitives/gossip-protocol/` — how membership info spreads when there's no central monitor.
- `02-distributed-primitives/lease/` — time-bounded ownership tokens that make fencing safe.
- `02-distributed-primitives/split-brain-and-fencing/` — what goes wrong, and how to stop it.